# OLA Review Sentiment Analysis using Machine Learning

The objective of this project is to analyze customer reviews of OLA ride services
and classify them into three sentiment categories: Positive, Neutral, and Negative.

# Problem Statement:
Customer feedback is important for improving service quality.
However, analyzing thousands of reviews manually is difficult and time-consuming.
This project aims to build a machine learning model that can automatically
identify the sentiment of customer reviews using Natural Language Processing (NLP).

## Solution Approach
We use text preprocessing, TF-IDF feature extraction, and multiple machine learning
algorithms to train a sentiment classification model.
The best-performing model is selected, tuned, and deployed.

#Import Required Libraries

In [ ]:
# Data Handling
import pandas as pd
import numpy as np

# Text Processing
import re
import nltk
from nltk.corpus import stopwords

# Machine Learning Utilities
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ML Models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Save & Load Model
import pickle


# Load Dataset and Explore Data

In [ ]:
# Load dataset
df = pd.read_csv("ola_review_dataset.csv")
df.head()


,review_id,rating,review_text
0,1e4c163e-144a-4ea4-b0bd-80e5e9f259bd,1,unexpected charges if driver cancel ride don't...
1,447aaf31-3968-45f1-877f-5e04d230606a,2,some problem with the app. Card is saved but c...
2,84e9cae0-5cb2-439f-8cd4-ba97fe72b478,3,Services of RAPIDO are far better.
3,2d50f89e-025a-4a29-88ab-00c5ee052bfe,1,Adding cancellation charges while the driver i...
4,49bcf932-c446-461a-ae58-a2c5f6d91fe6,1,very costly - uber is good


In [ ]:
pd.set_option("display.max_colwidth", None)
df.head(8)

,review_id,rating,review_text
0,1e4c163e-144a-4ea4-b0bd-80e5e9f259bd,1,unexpected charges if driver cancel ride don't use this app for booking ride
1,447aaf31-3968-45f1-877f-5e04d230606a,2,"some problem with the app. Card is saved but can't pay after the ride, it says, ""something went wrong"", and if you go to customer support, no options. Irony is no customer care connect apart from that. And then you can't book next ride. Stuck. Poor app."
2,84e9cae0-5cb2-439f-8cd4-ba97fe72b478,3,Services of RAPIDO are far better.
3,2d50f89e-025a-4a29-88ab-00c5ee052bfe,1,Adding cancellation charges while the driver is not responding. The support chat doesn't give option to express our side or chat with a representative. Asks us to send email and there also only automated response. No way to call and talk to customer care. Adding the CRN : 17068165. PS i do not have twitter or Facebook ids.
4,49bcf932-c446-461a-ae58-a2c5f6d91fe6,1,very costly - uber is good
5,10acd7e3-904f-4401-9165-8a9ad5bdab93,1,very bad service
6,204639f7-4213-4a5d-953e-8e94badca2c4,1,The worst app ever. Drivers used to ask rate 1st and then they neither come nor cancell and if you cancell then Ola wont refund you. They will ask to send mail and no proper reply rather just say we wont refund. so be careful when you will book through OLA.
7,13242f4e-66b1-48f6-a7b0-3cb43a498ff7,2,The ride prices are way too high for the minimal distances than the other platforms.


In [ ]:
# Check basic information
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    10000 non-null  object
 1   rating       10000 non-null  int64 
 2   review_text  10000 non-null  object
dtypes: int64(1), object(2)
memory usage: 234.5+ KB


In [ ]:
# Check shape (rows, columns)
print(df.shape)


(10000, 3)


In [ ]:
# Check missing values
print(df.isnull().sum())


review_id      0
rating         0
review_text    0
dtype: int64


In [ ]:
# Check rating distribution

print(df["rating"].value_counts())

rating
1    6926
5    1897
2     446
3     383
4     348
Name: count, dtype: int64


“Initially, the dataset was explored to understand its structure, detect missing values, and analyze rating distribution.”

#  Convert Ratings into Sentiment Labels

In [ ]:
# Function to convert rating to sentiment
def rating_to_sentiment(rating):
    if rating <= 2:
        return "Negative"
    elif rating == 3:
        return "Neutral"
    else:
        return "Positive"

# Apply function to create target column
df["sentiment"] = df["rating"].apply(rating_to_sentiment)

# Check sentiment distribution
print(df["sentiment"].value_counts())


sentiment
Negative    7372
Positive    2245
Neutral      383
Name: count, dtype: int64


In [ ]:
df["sentiment"].value_counts(normalize=True)*100

,proportion
sentiment,
Negative,73.72
Positive,22.45
Neutral,3.83


Ratings were converted into sentiment categories (Negative, Neutral, Positive) to form the target variable.

#  Text Cleaning and Preprocessing

In [ ]:
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# Function to clean text
def clean_text(text):

    text = text.lower()# Convert to lowercase

    text = re.sub(r"[^a-z\s]", "", text)# Remove numbers and special characters

    text = re.sub(r"\s+", " ", text)# Remove extra spaces

    words = text.split()# Tokenize (split into words)

    words = [word for word in words if word not in stop_words] # Remove stopwords

    return " ".join(words)# Join words back


# Apply cleaning on review text
df["clean_review"] = df["review_text"].apply(clean_text)

In [ ]:
# Show sample before and after cleaning
df[["review_text", "clean_review"]].head()

,review_text,clean_review
0,unexpected charges if driver cancel ride don't use this app for booking ride,unexpected charges driver cancel ride dont use app booking ride
1,"some problem with the app. Card is saved but can't pay after the ride, it says, ""something went wrong"", and if you go to customer support, no options. Irony is no customer care connect apart from that. And then you can't book next ride. Stuck. Poor app.",problem app card saved cant pay ride says something went wrong go customer support options irony customer care connect apart cant book next ride stuck poor app
2,Services of RAPIDO are far better.,services rapido far better
3,Adding cancellation charges while the driver is not responding. The support chat doesn't give option to express our side or chat with a representative. Asks us to send email and there also only automated response. No way to call and talk to customer care. Adding the CRN : 17068165. PS i do not have twitter or Facebook ids.,adding cancellation charges driver responding support chat doesnt give option express side chat representative asks us send email also automated response way call talk customer care adding crn ps twitter facebook ids
4,very costly - uber is good,costly uber good


Text preprocessing included lowercasing, removal of special characters, stopword removal, and whitespace normalization

#  Define Features (X), Target (y) and Split Data

In [ ]:
# Input feature (cleaned reviews) and Target variable (sentiment)
X = df["clean_review"]
y = df["sentiment"]

# Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% test data
    random_state=42,     # For reproducibility
    stratify=y           # Maintain class balance
)

In [ ]:
# Check split size
print("Training Data Size:", X_train.shape)
print("Testing Data Size :", X_test.shape)

Training Data Size: (8000,)
Testing Data Size : (2000,)


The dataset was divided into training (80%) and testing (20%) sets using stratified sampling.

#  Text Vectorization using TF-IDF

In [ ]:
# Initialize TF-IDF Vectorizer with unigrams and bigrams
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

# Fit on training data and transform
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test data (no fitting to avoid data leakage)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
X_train_tfidf[0]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 16 stored elements and shape (1, 5000)>

In [ ]:
# Check shape of TF-IDF matrices
print("TF-IDF Train Shape:", X_train_tfidf.shape)
print("TF-IDF Test Shape :", X_test_tfidf.shape)

TF-IDF Train Shape: (8000, 5000)
TF-IDF Test Shape : (2000, 5000)


Text data was converted into numerical features using TF-IDF with n-grams.

 # Train Multiple Machine Learning Models

KNN CLASSIFIER

In [ ]:
#KNN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_tfidf, y_train)

y_test_pred_knn = knn_model.predict(X_test_tfidf)

print("KNN Accuracy:", accuracy_score(y_test, y_test_pred_knn))

KNN Accuracy: 0.754


Naive Bayes

In [ ]:
modelnb = MultinomialNB()
modelnb.fit(X_train_tfidf, y_train)

MultinomialNB()

In [ ]:
y_train_pred = modelnb.predict(X_train_tfidf)
y_test_pred = modelnb.predict(X_test_tfidf)
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
print("Testing Accuracy :", accuracy_score(y_test, y_test_pred))


Training Accuracy: 0.899625
Testing Accuracy : 0.8975


In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Confusion Matrix:
[[1461    0   13]
 [  72    0    5]
 [ 115    0  334]]


Logistic Regression

In [ ]:
log_model = LogisticRegression()

log_model.fit(X_train_tfidf, y_train)

LogisticRegression()

In [ ]:
y_train_pred_lr = log_model.predict(X_train_tfidf)
y_test_pred_lr = log_model.predict(X_test_tfidf)

In [ ]:
print("Logistic Regression Train Accuracy:", accuracy_score(y_train, y_train_pred_lr))
print("Logistic Regression Test Accuracy :", accuracy_score(y_test, y_test_pred_lr))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred_lr))

Logistic Regression Train Accuracy: 0.918
Logistic Regression Test Accuracy : 0.9085

Confusion Matrix:
[[1450    0   24]
 [  70    0    7]
 [  82    0  367]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.91      0.98      0.94      1474
     Neutral       0.00      0.00      0.00        77
    Positive       0.92      0.82      0.87       449

    accuracy                           0.91      2000
   macro avg       0.61      0.60      0.60      2000
weighted avg       0.87      0.91      0.89      2000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


SVM

In [ ]:
from sklearn.svm import LinearSVC
svm_model = LinearSVC(max_iter=1000, class_weight="balanced")
svm_model.fit(X_train_tfidf, y_train)


LinearSVC(class_weight='balanced')

In [ ]:
y_test_pred_svm = svm_model.predict(X_test_tfidf)
print("SVM Accuracy:", accuracy_score(y_test, y_test_pred_svm))

SVM Accuracy: 0.876


Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt_model = DecisionTreeClassifier(
    max_depth=20,
    min_samples_split=10,
    random_state=42
)

dt_model.fit(X_train_tfidf, y_train)

y_test_pred_dt = dt_model.predict(X_test_tfidf)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_test_pred_dt))

Decision Tree Accuracy: 0.8665


Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42
)

rf_model.fit(X_train_tfidf, y_train)

y_test_pred_rf = rf_model.predict(X_test_tfidf)

print("Random Forest Accuracy:", accuracy_score(y_test, y_test_pred_rf))

Random Forest Accuracy: 0.8165


Multiple machine learning models were trained and compared for sentiment classification.

#  Hyperparameter Tuning using GridSearchCV

In [ ]:
# Base Logistic Regression model
lr_model = LogisticRegression(solver="liblinear")

# Parameters to tune
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "class_weight": [None, "balanced"],
    "max_iter": [500, 1000]
}

# GridSearch with 5-Fold Cross Validation
grid_search = GridSearchCV(
    lr_model,
    param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1
)

print("Running GridSearchCV...")

grid_search.fit(X_train_tfidf, y_train)

# Display best parameters
print("\nBest Parameters Found:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Score:")
print(grid_search.best_score_)


Running GridSearchCV...

Best Parameters Found:
{'C': 1, 'class_weight': 'balanced', 'max_iter': 500}

Best Cross-Validation Score:
0.8818794218517901


Hyperparameter tuning was performed using GridSearchCV with 5-fold cross-validation to optimize Logistic Regression.

 # Train Final Model Using Best Parameters

In [ ]:
# Create final Logistic Regression model
best_params = grid_search.best_params_

final_model = LogisticRegression(
    C=best_params["C"],
    class_weight=best_params["class_weight"],
    max_iter=best_params["max_iter"],
    solver="liblinear"
)

# Train final model on training data
final_model.fit(X_train_tfidf, y_train)

LogisticRegression(C=1, class_weight='balanced', max_iter=500,
                   solver='liblinear')

The final model was trained using the optimal hyperparameters obtained from GridSearchCV.

#Evaluate Final Model Performance

In [ ]:
# Predict on training data AND testing data
y_train_pred = final_model.predict(X_train_tfidf)
y_test_pred = final_model.predict(X_test_tfidf)


train_acc = accuracy_score(y_train, y_train_pred)
print("Training Accuracy:", train_acc)
test_acc = accuracy_score(y_test, y_test_pred)
print("Testing Accuracy :", test_acc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))


Training Accuracy: 0.936125
Testing Accuracy : 0.89

Confusion Matrix:
[[1404   38   32]
 [  61    9    7]
 [  64   18  367]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.92      0.95      0.94      1474
     Neutral       0.14      0.12      0.13        77
    Positive       0.90      0.82      0.86       449

    accuracy                           0.89      2000
   macro avg       0.65      0.63      0.64      2000
weighted avg       0.89      0.89      0.89      2000



The final model was evaluated using accuracy, confusion matrix, and classification report.

In [ ]:
#  Manual Testing
# Function to predict sentiment for new review
def predict_review(review_text):

    cleaned = clean_text(review_text)
    vector = tfidf.transform([cleaned])
    prediction = final_model.predict(vector)[0]

    return prediction


# Test with sample reviews
test_reviews = [
    "driver was very polite and helpful",
    "average ride nothing special",
    "worst experience ever",
    "nice car and smooth ride"
]

# Display predictions
print("\nManual Testing Results:\n")

for review in test_reviews:

    result = predict_review(review)

    print("Review:", review)
    print("Prediction:", result)
    print("-" * 40)



Manual Testing Results:

Review: driver was very polite and helpful
Prediction: Positive
----------------------------------------
Review: average ride nothing special
Prediction: Negative
----------------------------------------
Review: worst experience ever
Prediction: Negative
----------------------------------------
Review: nice car and smooth ride
Prediction: Positive
----------------------------------------


In [ ]:
#  Save Model and Vectorizer for Deployment


# Save trained model
with open("final_logistic_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

# Save TF-IDF vectorizer
with open("final_tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("Model and Vectorizer saved successfully!")


Model and Vectorizer saved successfully!


# Conclusion

In this project, sentiment analysis was performed on OLA customer reviews using
Natural Language Processing and Machine Learning techniques.

The data was preprocessed by cleaning text, removing stopwords, and normalizing input.
TF-IDF was used for feature extraction with unigrams and bigrams.

Multiple machine learning models were trained and compared, including Naive Bayes,
Logistic Regression, SVM, Decision Tree, Random Forest, KNN, and Ridge Classifier.

Logistic Regression was selected as the final model after hyperparameter tuning
using GridSearchCV, as it provided stable and balanced performance.

The final model achieved good accuracy and was successfully deployed using
Hugging Face Spaces for real-time prediction.

This system can help businesses analyze customer feedback and improve service quality.